# 067 — Reconocimiento automático del habla

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


## 📖 Resumen de la materia

**Señal → features:** audio muestreado a 16 kHz → tramas de 25 ms (hop 10 ms) → FFT →
filtros en escala **mel** + log → log-mel espectrograma (entrada de Whisper, 80 bandas).
Los **MFCC** (DCT sobre el log-mel) fueron el estándar de la era HMM.

**Alineación:** ~300 tramas para ~10 palabras. **CTC** entrena sin alineación explícita:
emite símbolo o blanco `∅` por trama y suma la probabilidad de todas las alineaciones que
colapsan a la transcripción. **Whisper** (2022) usa en cambio un transformer
encoder-decoder que genera texto token a token, entrenado con 680 000 h de supervisión
débil: robusto sin fine-tuning, pero autoregresivo (lento) y capaz de alucinar texto en
silencios.

**Métrica:** `WER = (S+D+I)/N` con alineación de Levenshtein por palabra; puede superar
100 % y debe desglosarse por subgrupo de hablantes (acentos, edad, género).


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Solución de referencia

**Ejercicio 1.** Alineación: `activa OK, la OK, alarma OK, de→a (S), las OK, siete OK`,
más `y` y `media` sobrantes (I=2). S=1, D=0, I=2 → `WER = 3/6 = 50 %`. Sí puede superar
100 %: ref = `hola` (N=1), hyp = `hola qué tal` → I=2, WER = 200 %.

**Ejercicio 2.** Muestras: `5 · 16 000 = 80 000`. Tramas:
`1 + (5000 − 25)/10 ≈ 498`. Entrada: matriz `≈498 × 80` — de 80 000 números crudos a una
representación tiempo-frecuencia diez veces más compacta y perceptualmente organizada.

**Ejercicio 3.** Colapso: quitar blancos y repeticiones adyacentes →
`h o l a`... cuidado: `l l l` colapsa a una sola `l`, así que el resultado es `hola`. Sin
blanco, `o o` y `l l l` seguirían colapsando (bien aquí), pero sería **imposible** emitir
letras dobles reales ("llama" → "lama"). En voz cada fonema dura muchas tramas (una vocal
larga puede ocupar 20+ tramas), de modo que las repeticiones masivas son la norma y el
mecanismo de colapso + blanco es imprescindible.

**Ejercicio 4.** Implementación debajo: reutiliza la misma programación dinámica del CER
(clase 063) sobre listas de palabras; imprime 0.5.


In [ ]:
result = run_lab("perception", seed=67)
assert result["kind"] == "perception"
assert result["evidence"]
show(result)


In [ ]:
# Ejercicio 4 — WER verificado con código
def levenshtein(r, h):
    m, n = len(r), len(h)
    d = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(m + 1):
        d[i][0] = i
    for j in range(n + 1):
        d[0][j] = j
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            costo = 0 if r[i - 1] == h[j - 1] else 1
            d[i][j] = min(d[i - 1][j] + 1, d[i][j - 1] + 1, d[i - 1][j - 1] + costo)
    return d[m][n]

def wer(ref, hyp):
    r, h = ref.split(), hyp.split()
    return levenshtein(r, h) / len(r)

print(wer("activa la alarma de las siete",
          "activa la alarma a las siete y media"))   # 0.5
print(wer("hola", "hola qué tal"))                    # 2.0 → WER 200 %


In [ ]:
# Ejercicios 2 y 3 — tramas y colapso CTC
muestras = 5 * 16000
tramas = 1 + (5000 - 25) // 10
print("muestras:", muestras, "| tramas:", tramas, "| entrada:", (tramas, 80))

def ctc_collapse(frames, blank="∅"):
    out, prev = [], None
    for f in frames:
        if f != prev and f != blank:
            out.append(f)
        prev = f
    return "".join(out)

print(ctc_collapse("∅ h ∅ o o ∅ l l l ∅ a a".split()))  # hola


## Reflexión

1. Tu ASR reporta WER 8 % global, pero el subtitulado falla sistemáticamente con hablantes
   andinos. ¿Qué desglose de evaluación faltó y qué datos corregirían el problema?
2. ¿Por qué la arquitectura autoregresiva de Whisper lo hace propenso a "transcribir"
   música o silencio, y qué componente previo del pipeline lo mitiga?
3. En una consulta médica transcrita, un WER de 5 % ¿es aceptable? ¿Qué palabras te
   preocupan más que el promedio y cómo las vigilarías?
